In [ ]:
import sys
from pathlib import Path
import subprocess

src_path = str(Path.cwd().parent / "src")
if src_path not in sys.path:
    sys.path.append(src_path)
from ini import initialisation 

In [ ]:
print("Current working directory:", os.getcwd())

PROJECT_ROOT = Path("..").resolve()

# Setze das Arbeitsverzeichnis auf das Hauptprojektverzeichnis
os.chdir(f"{PROJECT_ROOT}")

# Überprüfe, ob das Verzeichnis korrekt gesetzt wurde
print("Current working directory:", os.getcwd())

In [ ]:
c = ColorManager().get_colors()

In [ ]:
df_oil = parquet_loader(Dataset.OIL)
df_items = parquet_loader(Dataset.ITEMS)
df_holidays = parquet_loader(Dataset.HOLIDAYS_EVENTS)
df_stores = parquet_loader(Dataset.STORES)
df_transactions = parquet_loader(Dataset.TRANSACTIONS)
df_train = parquet_loader(Dataset.TRAIN)

In [ ]:
#Visualize data set
for element in Dataset:
    print(element, "\n", parquet_loader(element).head(), "\n")

In [ ]:
# OIL VS SALES
df_oil["date"] = pd.to_datetime(df_oil["date"])
sales_oil = (
    df_train.groupby("date")["unit_sales"].sum().reset_index()
)  # Aggregate daily sales
sales_oil = sales_oil.merge(df_oil, on="date", how="left")

fig = go.Figure()

# Unit Sales (linke y-Achse)
fig.add_trace(
    go.Scatter(
        x=sales_oil["date"],
        y=sales_oil["unit_sales"],
        name="Total Unit Sales",
        mode="lines",
        opacity=1.0,
        yaxis="y1",
    )
)

# Oil Prices (rechte y-Achse)
fig.add_trace(
    go.Scatter(
        x=sales_oil["date"],
        y=sales_oil["dcoilwtico"],
        name="Oil Prices",
        mode="lines",
        opacity=1.0,
        yaxis="y2",
    )
)

fig.update_layout(
    title="Daily Sales vs Oil Prices",
    xaxis={"title": "Date"},
    yaxis={
        "title": "Total Unit Sales",
        "side": "left",
    },
    yaxis2={
        "title": "Oil Prices",
        "overlaying": "y",
        "side": "right",
    },
    legend={"x": 0.01, "y": 0.99},
)

fig.show()
save_all(fig, "eda/daily_sales_vs_oil_prices")  # (overwrite = True)

In [ ]:
## STORE SALES
store_sales = df_train.groupby("store_nbr")["unit_sales"].sum().reset_index()

# Sort stores by sales
store_sales = store_sales.sort_values(by="unit_sales", ascending=False)

store_sales = store_sales.copy()
store_sales["category"] = "Other"

top5_idx = store_sales.nlargest(5, "unit_sales").index
bottom5_idx = store_sales.nsmallest(5, "unit_sales").index

store_sales.loc[top5_idx, "category"] = "Top 5"
store_sales.loc[bottom5_idx, "category"] = "Bottom 5"

fig = px.bar(
    store_sales,
    x="store_nbr",
    y="unit_sales",
    color="category",
    title="Total Unit Sales Per Store (Top 5 & Bottom 5 Highlighted)",
    color_discrete_map={
        "Top 5": c.forecast,
        "Bottom 5": c.anomaly,
        "Other": c.border,
    },
)

fig.update_layout(
    xaxis_title="Store Number",
    yaxis_title="Total Sales",
    xaxis_tickangle=-90,
)

fig.show()

In [ ]:
## MONTHLY SALES ACROSS YEARS

df_train["year"] = df_train["date"].dt.year
df_train["month"] = df_train["date"].dt.month

monthly_sales_by_year = (
    df_train.groupby(["year", "month"])["unit_sales"].sum().reset_index()
)


fig = px.line(
    monthly_sales_by_year,
    x="month",
    y="unit_sales",
    color="year",
    markers=True,
    title="Monthly Sales Trend Across Years",
)

fig.update_layout(
    xaxis={
        "title": "Month",
        "tickmode": "linear",
        "tick0": 1,
        "dtick": 1,
    },
    yaxis={"title": "Total Units Sold"},
    legend_title_text="Year",
)

fig.show()

In [ ]:
# CONSECUTIVE MONTHLY SALES OVER YEARS

#  Create a sequential month-year column for better visualization
df_train["year_month"] = df_train["date"].dt.to_period("M")  # Format: YYYY-MM

# Aggregate sales by year-month
monthly_sales = df_train.groupby("year_month")["unit_sales"].sum().reset_index()

# Convert year_month to string for plotting
monthly_sales["year_month"] = monthly_sales["year_month"].astype(str)

monthly_sales_plot = monthly_sales.copy()
monthly_sales_plot["year_month"] = pd.to_datetime(monthly_sales_plot["year_month"])

fig = px.line(
    monthly_sales_plot,
    x="year_month",
    y="unit_sales",
    markers=True,
    title="Consecutive Monthly Sales Trend Over Years",
)

fig.update_layout(
    xaxis_title="Year-Month",
    yaxis_title="Total Sales",
)

fig.show()

In [ ]:
# MONTHLY SEASONALITY IN SALES
daily_sales = df_train.groupby("date")["unit_sales"].sum().reset_index()
daily_sales["rolling_avg"] = daily_sales["unit_sales"].rolling(window=30).mean()
daily_sales["month"] = daily_sales["date"].dt.month

# Aggregate sales by month
monthly_sales = daily_sales.groupby("month")["unit_sales"].mean().reset_index()

fig = px.line(
    monthly_sales,
    x="month",
    y="unit_sales",
    markers=True,
    title="Monthly Seasonality in Sales",
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Average Sales",
    xaxis={
        "tickmode": "array",
        "tickvals": list(range(1, 13)),
        "ticktext": [
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
        ],
    },
)


fig.update_yaxes(
    tickformat=",.0f",
    exponentformat="none",
    showexponent="none",
)

fig.show()

In [ ]:
# WEEKLY SEASONALITY IN SALES
# Extract day of the week from the date (Monday=0, Sunday=6)
daily_sales["day_of_week"] = daily_sales["date"].dt.dayofweek

# Aggregate sales by day of the week
weekly_sales = daily_sales.groupby("day_of_week")["unit_sales"].mean().reset_index()

fig = px.line(
    weekly_sales,
    x="day_of_week",
    y="unit_sales",
    markers=True,
    title="Weekly Seasonality in Sales",
)

fig.update_layout(
    xaxis_title="Day of Week",
    yaxis_title="Average Sales",
    xaxis={
        "tickmode": "array",
        "tickvals": list(range(7)),
        "ticktext": ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    },
)

# ✔️ Fix für große Zahlen (lesbare Achse)
fig.update_yaxes(
    tickformat=",.0f",
    exponentformat="none",
    showexponent="none",
)

fig.show()

In [ ]:
# SEASONAL DECOMPOSITION
from statsmodels.tsa.seasonal import seasonal_decompose

# Zeitreihe vorbereiten (DatetimeIndex + sortiert)
# WICHTIG: seasonal_decompose benötigt oft eine gesetzte Frequenz im Index
ts = daily_sales.set_index("date")["unit_sales"].sort_index()

# Falls Lücken existieren, füllen wir diese (seasonal_decompose erlaubt keine NaNs)
if ts.index.freq is None:
    ts = ts.asfreq("D").interpolate().fillna(0)

# Decompose the time series (Einmalige Berechnung auf der vorbereiteten Reihe)
decomp = seasonal_decompose(ts, model="additive", period=365)

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    subplot_titles=("<b>Observed (Unit Sales)</b>", "<b>Trend</b>", "<b>Seasonal</b>", "<b>Residual</b>"),
)

fig.add_trace(
    go.Scatter(x=ts.index, y=decomp.observed, mode="lines", name="Observed"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=ts.index, y=decomp.trend, mode="lines", name="Trend"), 
    row=2, 
    col=1
)
fig.add_trace(
    go.Scatter(x=ts.index, y=decomp.seasonal, mode="lines", name="Seasonal"),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(x=ts.index, y=decomp.resid, mode="lines", name="Residual"), 
    row=4, 
    col=1
)

fig.update_layout(
    height=900,
    title="Seasonal Decomposition (Additive, period=365)",
    showlegend=False,
    margin={"t": 100, "b": 50}
)

# Große Zahlen besser lesbar formatieren
fig.update_yaxes(tickformat=",.0f", exponentformat="none", row=1, col=1)
fig.update_yaxes(tickformat=",.0f", exponentformat="none", row=2, col=1)

# Integration deines Design-Systems
if 'apply_modern_theme' in globals():
    apply_modern_theme(fig)

fig.show()

# Sicherer Export (falls save_all definiert ist)
try:
    save_all(fig, "img/analysis/seasonal_decomposition")
except NameError:
    pass

In [ ]:
# OPTIMIERTE DATEN-VORBEREITUNG
df = df_train.copy()
df["date"] = pd.to_datetime(df["date"], format='%Y-%m-%d')
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.to_period("M")
df["week"] = df["date"].dt.to_period("W")
df["dow"] = df["date"].dt.dayofweek

print(f"✅ Konvertierung abgeschlossen. Shape: {df.shape}")
print(df[["date", "year", "month","week", "dow"]].head())

In [ ]:
df.head()

In [ ]:
# df.isna().mean().sort_values(ascending=False)
# df['unit_sales'].describe()
(df["unit_sales"] < 0).sum()

Dimensionen & Abdeckung

➡️ Welche Stores / Items haben kurze Historien?

In [ ]:
# df['store_nbr'].nunique()
# df['item_nbr'].nunique()
# df['date'].nunique()

df.groupby("store_nbr")["date"].agg(["min", "max"])

54 Stores 
    spätere Eröffnungen: 
    
    •	store 53 → 2014-05-29
	•	store 20 → 2015-02-13
    •	store 29 → 2015-03-20
	•	store 21 → 2015-07-24
    •	store 42 → 2015-08-21
	•	store 22 → 2015-10-09
	•	store 52 → 2017-04-20
	
4036 Items 

Store-Level EDA

In [ ]:
store_daily = df.groupby(["store_nbr", "date"])["unit_sales"].sum().reset_index()
store_weekly = (
    df.groupby(["store_nbr", "year", "week"])["unit_sales"].sum().reset_index()
)
store_monthly = df.groupby(["store_nbr", "month"])["unit_sales"].sum()

In [ ]:
# DAYLY SALES FOR A SPECIFIC STORE
store_id = 2

df_store = store_daily[store_daily["store_nbr"] == store_id]

fig = px.line(
    df_store, x="date", y="unit_sales", title=f"Daily Unit Sales - Store {store_id}"
)

fig.show()

In [ ]:
## STORE x DATE HEATMAP
# optional: auf ein Jahr beschränken
# df_2017 = store_daily[
#    store_daily["date"].dt.year == 2017
# ]

pivot = df.pivot_table(index="store_nbr", columns="date", values="unit_sales")

fig = px.imshow(
    pivot, aspect="auto", title="Store x Date Heatmap", color_continuous_scale="Viridis"
)

fig.show()

Verteilung der täglichen Umsätze 
👉 erkennt Stores mit hoher Volatilität

In [ ]:
## DAYLY UNIT SALES DISTRIBUTION PER STORE
fig = px.box(
    store_daily,
    x="store_nbr",
    y="unit_sales",
    title="Distribution of Daily Unit Sales per Store",
)

fig.show()

# Durchschnittlicher Wochenverlauf 

In [ ]:
df.head()

In [ ]:
store_daily["week"] = store_daily["date"].dt.isocalendar().week

fig = px.box(
    store_daily,
    x="week",
    y="unit_sales",
    title="Unit Sales by Calendar Week of Year",
)

fig.show()

In [ ]:
total_daily = store_daily.groupby("date")["unit_sales"].sum().reset_index()

fig = px.line(
    total_daily, x="date", y="unit_sales", title="Total Daily Unit Sales (All Stores)"
)

fig.show()

Item-Level EDA

In [ ]:
item_daily = df.groupby(["item_nbr", "date"])["unit_sales"].sum().reset_index()
item_weekly = df.groupby(["item_nbr", "year", "week"])["unit_sales"].sum().reset_index()
item_monthly = df.groupby(["item_nbr", "month"])["unit_sales"].sum()

In [ ]:
# DAYLY SALES FOR A SPECIFIC ITEM
item_id = 103665  # Beispiel

df_item = item_daily[item_daily["item_nbr"] == item_id]

fig = px.line(
    df_item, x="date", y="unit_sales", title=f"Daily Unit Sales - Item {item_id}"
)
fig.show()

Nachfrageverteilung pro Item

In [ ]:
fig = px.box(
    item_daily,
    x="item_nbr",
    y="unit_sales",
    title="Distribution of Daily Unit Sales per Item",
)
fig.show()

In [ ]:
fig = px.histogram(
    df_item, x="unit_sales", nbins=50, title=f"Unit Sales Distribution - Item {item_id}"
)
fig.show()

In [ ]:
item_daily["dow"] = item_daily["date"].dt.day_name()

fig = px.box(
    item_daily,
    x="dow",
    y="unit_sales",
    title="Unit Sales by Day of Week",
    category_orders={
        "dow": [
            "Monday",
            "Tuesday",
            "Wednesday",
            "Thursday",
            "Friday",
            "Saturday",
            "Sunday",
        ]
    },
)
fig.show()

In [ ]:
item_daily["week"] = item_daily["date"].dt.isocalendar().week

fig = px.box(
    item_daily, x="week", y="unit_sales", title="Unit Sales by Calendar Week of Year"
)
fig.show()

In [ ]:
top_items = (
    item_daily.groupby("item_nbr", as_index=False)["unit_sales"]
    .sum()
    .sort_values("unit_sales", ascending=False)
    .head(20)
)

fig = px.bar(
    top_items,
    x="item_nbr",
    y="unit_sales",
    title="Top 20 Items by Total Unit Sales",
    text="item_nbr",  # absolute Item-Nummern auf den Balken
)

fig.update_layout(
    xaxis={
        "type": "category",  # verhindert automatische numerische Skalierung
        "categoryorder": "total descending",  # gleiche Reihenfolge wie die Top 20
    }
)

fig.update_traces(
    textposition="outside"  # Beschriftung gut lesbar
)

fig.show()

Store/Item-EDA 


In [ ]:
store_item = df.groupby(["store_nbr", "item_nbr"])["unit_sales"].sum().reset_index()
store_item.head()

Top Items pro Store 

In [ ]:
store_id = 1

top_items_store = (
    store_item[store_item["store_nbr"] == store_id]
    .sort_values("unit_sales", ascending=False)
    .head(20)
)

fig = px.bar(
    top_items_store,
    x="item_nbr",
    y="unit_sales",
    title=f"Top 20 Items - Store {store_id}",
    text="item_nbr",  # absolute Itemnummern anzeigen
)

fig.update_layout(
    xaxis={
        "type": "category",  # erzwingt nur diese 20 Items
        "categoryorder": "total descending",
    }
)

fig.update_traces(textposition="outside")

fig.show()

In [ ]:
# Share of Sales Top10
def top_k_share(df, k=10):
    return (
        df.sort_values("unit_sales", ascending=False).head(k)["unit_sales"].sum()
        / df["unit_sales"].sum()
    )


concentration = (
    store_item.groupby("store_nbr").apply(top_k_share).reset_index(name="top10_share")
)

fig = px.bar(
    concentration,
    x="store_nbr",
    y="top10_share",
    title="Share of Sales from Top 10 Items per Store",
)
fig.show()

In [ ]:
# Distribution Items
fig = px.box(
    store_item,
    x="store_nbr",
    y="unit_sales",
    title="Distribution of Item Sales per Store",
)
fig.update_yaxes(type="log")
fig.show()

Zero-Rate 

In [ ]:
# PERIOD SUMMEN
store_item_daily = (df_train.groupby(["store_nbr", "item_nbr", "date"])[["unit_sales", "onpromotion"]].sum().reset_index())
store_item_daily["year"] = store_item_daily["date"].dt.year
store_item_daily["dow"] = store_item_daily["date"].dt.dayofweek

print(f"✅ Aggregation erfolgreich. Shape: {store_item_daily.shape}")

In [ ]:
store_item_daily = store_item_daily[["date", "store_nbr", "item_nbr", "unit_sales"]]

In [ ]:
store_item_daily.head()

✅ Zentrale Ausschluss-Kriterien (empfohlen)

1️⃣ Anzahl Verkaufstage (wichtigstes Kriterium)

Wie oft wurde überhaupt verkauft?

🔴 Ausschließen, wenn:  n_sales_days < 30  👉 Weniger als ~30 Verkaufstage → kein stabiles Muster 

In [ ]:
sales_days = (
    store_item_daily.groupby(["store_nbr", "item_nbr"])["unit_sales"]
    .apply(lambda x: (x > 0).sum())
    .reset_index(name="n_sales_days")
)

# Alternative (noch schneller für 125M Zeilen):
# sales_days = (
#     store_item_daily.assign(has_sales=store_item_daily["unit_sales"] > 0)
#     .groupby(["store_nbr", "item_nbr"])["has_sales"]
#     .sum()
#     .reset_index(name="n_sales_days")
# )

print("✅ 'n_sales_days' erfolgreich berechnet (ohne Warnings).")
print(sales_days.head())

2️⃣ Zero-Rate (Intermittency)

Wie oft ist die Nachfrage null?

🔴 Ausschließen, wenn: zero_rate > 0.6  👉 Mehr als 60 % Zero-Tage → intermittente Nachfrage

In [ ]:
# PERFORMANCE-OPTIMIERTE ZERO-RATE BERECHNUNG

df = store_item_daily.copy()
# Datum sicherheitshalber konvertieren (falls nicht schon geschehen)
df["date"] = pd.to_datetime(df["date"])

# 1) Basis: Tägliche Sales (nutzen wir direkt, um Lücken zu zählen)
# Wir gehen davon aus, dass store_item_daily bereits aggregiert ist.
daily = df.groupby(["store_nbr", "item_nbr", "date"], as_index=False)["unit_sales"].sum()

# 2) Vollständigen Zeitraum bestimmen (Anzahl der Tage)
min_date = daily["date"].min()
max_date = daily["date"].max()
total_days_in_period = (max_date - min_date).days + 1

# 3) Tatsächliche Verkaufstage pro Paar zählen
# Jede Zeile in 'daily' repräsentiert einen Tag mit unit_sales (auch wenn diese 0 wären)
# Wir zählen hier nur Tage, an denen WIRKLICH unit_sales > 0 stattfand.
sales_counts = (
    daily[daily["unit_sales"] > 0]
    .groupby(["store_nbr", "item_nbr"])["date"]
    .count()
    .reset_index(name="active_sales_days")
)

# 4) Alle beobachteten store-item Kombis sicherstellen
pairs = daily[["store_nbr", "item_nbr"]].drop_duplicates()

# 5) Zero-Rate mathematisch berechnen statt durch Grid-Expansion
# Zero_Rate = (Gesamttage - Tage mit Verkäufen) / Gesamttage
zero_rate = pairs.merge(sales_counts, on=["store_nbr", "item_nbr"], how="left")
zero_rate["active_sales_days"] = zero_rate["active_sales_days"].fillna(0)

zero_rate["zero_rate"] = (total_days_in_period - zero_rate["active_sales_days"]) / total_days_in_period

# Finales Resultat aufräumen
zero_rate = zero_rate[["store_nbr", "item_nbr", "zero_rate"]]

print(f"✅ Zero-Rate berechnet für {len(zero_rate)} Paare über einen Zeitraum von {total_days_in_period} Tagen.")
print(zero_rate.head())

3️⃣ Gesamtvolumen

Wie relevant ist das Item überhaupt?

🔴 Ausschließen, wenn: total_sales < 50 👉 Sonst überfitten Modelle auf Rauschen

In [ ]:
total_sales = (
    store_item_daily.groupby(["store_nbr", "item_nbr"])["unit_sales"]
    .sum()
    .reset_index(name="total_sales")
)
print(f"✅ Gesamtvolumen berechnet für {len(total_sales)}")
print(total_sales.head())

4️⃣ Zeitliche Abdeckung

Über welchen Zeitraum gibt es Daten?

🔴 Ausschließen, wenn:  < 90 Tage Daten

In [ ]:
coverage = (
    store_item_daily.groupby(["store_nbr", "item_nbr"])
    .agg(first_day=("date", "min"), last_day=("date", "max"))
    .reset_index()
)
print(f"✅ Gesamtvolumen berechnet für {len(coverage)}")
print(coverage.head())

🧮 Kombinierter Quality-Score (sehr empfohlen)
        👉 Nur forecastable == True prognostizieren 

In [ ]:
quality = sales_days.merge(zero_rate, on=["store_nbr", "item_nbr"]).merge(
    total_sales, on=["store_nbr", "item_nbr"]
)

quality["forecastable"] = (
    (quality["n_sales_days"] >= 30)
    & (quality["zero_rate"] <= 0.6)
    & (quality["total_sales"] >= 50)
)
print(f"✅ Gesamtvolumen berechnet für {len(quality)}")
print(quality.head())

📊 Welche Visualisierungen machen Sinn?

1️⃣ Scatter: Zero-Rate vs. Sales-Tage (Pflicht)

📌 Beste Übersicht
	•	x = n_sales_days
	•	y = zero_rate
	•	Farbe = forecastable

👉 Trennung sofort sichtbar

2️⃣ Histogramme (Grenzwerte validieren)

a) Zero-Rate
	•	Wo liegt die Masse?
	•	Wo wird’s problematisch?

b) n_sales_days
	•	Wie viele Items sind „tot“?

3️⃣ Bubble-Plot (optional, aber stark)
	•	x = n_sales_days
	•	y = zero_rate
	•	Größe = total_sales
👉 Zeigt:
	•	seltene Bestseller
	•	häufige Low-Volume Items

4️⃣ Time-Series Beispiele (qualitativ!)

Zeige:
	•	1 „gutes“
	•	1 „grenzwertiges“
	•	1 „schlechtes“

👉 Super für Stakeholder & App-UX

In [ ]:
# Vorbereitung (robust: IDs als String für saubere Achsen/Labels)

quality_plot = quality.copy()
quality_plot["store_nbr"] = quality_plot["store_nbr"].astype(str)
quality_plot["item_nbr"] = quality_plot["item_nbr"].astype(str)
quality_plot["key"] = quality_plot["store_nbr"] + " x " + quality_plot["item_nbr"]

In [ ]:
# 1) Scatter: Zero-Rate vs Verkaufstage (Pflicht-Plot)

fig = px.scatter(
    quality_plot,
    x="n_sales_days",
    y="zero_rate",
    color="forecastable",
    hover_name="key",
    hover_data={"total_sales": True, "n_sales_days": True, "zero_rate": ":.2%"},
    title="Forecast-Eignung: Zero-Rate vs. Anzahl Verkaufstage",
)

fig.update_yaxes(tickformat=".0%")
fig.update_layout(
    xaxis_title="Anzahl Tage mit Verkäufen (unit_sales > 0)",
    yaxis_title="Zero-Rate (Anteil Tage mit unit_sales = 0)",
)
fig.show()

In [ ]:
item_sales = (
    store_item_daily[store_item_daily["unit_sales"] > 0]
    .drop_duplicates(["date", "item_nbr"])
    .assign(sold=1)[["date", "item_nbr", "sold"]]
)

In [ ]:
items_by_days_sold = (
    store_item_daily.loc[store_item_daily["unit_sales"] > 0]
    .groupby("item_nbr")["date"]
    .nunique()
    .reset_index(name="days_sold")
    .sort_values("days_sold", ascending=False)
)

# optional: als int
items_by_days_sold["days_sold"] = items_by_days_sold["days_sold"].astype("int32")

In [ ]:
items_by_days_sold.tail(50)

In [ ]:
fig = px.histogram(
    items_by_days_sold,
    x="days_sold",
    nbins=60,
    title="Distribution of Days Sold per Item",
    labels={"days_sold": "Days sold (unique dates with sales > 0)"},
)

fig.update_layout(yaxis_title="Number of Items")
fig.update_yaxes(type="log")  # <- wichtiger als log-x

fig.show()

In [ ]:
sold = store_item_daily.loc[
    store_item_daily["unit_sales"] > 0, ["date", "store_nbr", "item_nbr", "unit_sales"]
]

In [ ]:
store_item_stats = (
    sold.groupby(["store_nbr", "item_nbr"])
    .agg(
        days_sold=("date", "nunique"),  # an wie vielen Tagen verkauft
        total_units=("unit_sales", "sum"),  # Gesamtmenge
    )
    .reset_index()
)

In [ ]:
days_cutoff = store_item_stats["days_sold"].quantile(0.10)
units_cutoff = store_item_stats["total_units"].quantile(0.10)

slow_movers = store_item_stats.query(
    "days_sold <= @days_cutoff and total_units <= @units_cutoff"
)

In [ ]:
slow_movers = slow_movers.sort_values(["days_sold", "total_units"])

slow_movers.head(20)

In [ ]:
slow_per_store = (
    slow_movers.groupby("store_nbr")
    .size()
    .reset_index(name="n_slow_items")
    .sort_values("n_slow_items", ascending=False)
)

In [ ]:
fig = px.scatter(
    store_item_stats,
    x="days_sold",
    y="total_units",
    hover_data=["store_nbr", "item_nbr"],
    title="Store-Item Sales Performance",
)

fig.update_xaxes(type="log")
fig.update_yaxes(type="log")

fig.show()

# ➡️ unten links = selten verkauft, geringe stückzahl

In [ ]:
# 1) Flags / Labels vorbereiten
df = store_item_stats.copy()

df["is_slow"] = (df["days_sold"] < 160) & (df["total_units"] < 50)
df["is_slow_str"] = df["is_slow"].map({True: "Slow", False: "Other"})

stores = sorted(df["store_nbr"].unique())

# 2) Figure mit Traces: pro Store zwei Traces (Slow + Other)
fig = go.Figure()

for s in stores:
    df_s = df[df["store_nbr"] == s]

    # CHIRURGISCHER FIX: Benutze ~ statt 'not' für Pandas Series Negierung
    df_other = df_s[~df_s["is_slow"]]
    df_slow = df_s[df_s["is_slow"]]

    # Other Trace
    fig.add_trace(
        go.Scatter(
            x=df_other["days_sold"],
            y=df_other["total_units"],
            mode="markers",
            name="Other",
            visible=(s == stores[0]),
            marker={"size": 6, "opacity": 0.7},
            text=df_other["item_nbr"],
            hovertemplate="Store %{customdata[0]}<br>Item %{text}<br>Days %{x}<br>Units %{y}<extra></extra>",
            customdata=[[s]] * len(df_other),
        )
    )

    # Slow Trace
    fig.add_trace(
        go.Scatter(
            x=df_slow["days_sold"],
            y=df_slow["total_units"],
            mode="markers",
            name="Slow",
            visible=(s == stores[0]),
            marker={"size": 7, "symbol": "x", "opacity": 0.8},
            text=df_slow["item_nbr"],
            hovertemplate="Store %{customdata[0]}<br>Item %{text}<br>Days %{x}<br>Units %{y}<extra></extra>",
            customdata=[[s]] * len(df_slow),
        )
    )

# 3) Dropdown Buttons: pro Store genau die 2 zugehörigen Traces sichtbar
buttons = []
n_traces_per_store = 2

for i, s in enumerate(stores):
    # Logik für Sichtbarkeit: Nur das aktuelle Paar auf True setzen
    visible = [False] * (len(stores) * n_traces_per_store)
    visible[i * 2] = True      # Other
    visible[i * 2 + 1] = True  # Slow

    buttons.append(
        {
            "label": f"Store {s}",
            "method": "update",
            "args": [
                {"visible": visible},
                {"title": f"Store-Item Sales Performance (Store {s})"},
            ],
        }
    )

# 4) Layout / Achsen
fig.update_layout(
    title=f"Store-Item Sales Performance (Store {stores[0]})",
    xaxis={"title": "Days Sold (log)", "type": "log"},
    yaxis={"title": "Total Units (log)", "type": "log"},
    updatemenus=[
        {
            "buttons": buttons,
            "direction": "down",
            "x": 0.01,
            "y": 1.15,
            "showactive": True,
            "active": 0
        }
    ],
    legend_title_text="Category",
    margin={"t": 100}
)

# Integration deines Design-Systems
if 'apply_modern_theme' in globals():
    apply_modern_theme(fig)

fig.show()

# Zentrale Datenfragen 

1. Hat die Serie genug Daten? 
	Frage: Wie viele nicht-null Tage pro Store×Item?
    		z.B.
	    •	< 90 Tage → raus
	    •	< 180 Tage → kritisch
	    •	365 Tage → gut


2. Ist die Serie nicht nur Nullen?
    Sparsity = #Tage mit sales>0 / #Tage total
        Typische Klassen:
	    •	0.7 → gut
	    •	0.3–0.7 → schwierig
	    •	< 0.3 → fast unmöglich


3. Ist die Serie stationär genug?
	Keine harte Stationarität, aber:
		•	gibt es lange Totphasen?
		•	gibt es nur einen kurzen Verkaufszeitraum?


Items

4. Item Lifecycle: Wann wurde das Item eingeführt? Wann ist es „gestorben“?
	Viele Items:
		•	werden nur 6–12 Monate verkauft
		•	danach nie wieder
			→ Forecast ist sinnlos.


5. Item Popularity: Gesamtabsatz Item über alle Stores?
	 Long-tail:
		•	30% der Items machen 2% Umsatz. Diese sind meist nicht forecastbar.


Stores

6. Store Stability
	Fragen:
		•	Hat der Store konstant Traffic?
		•	Gibt es strukturelle Brüche?

Transactions

Extrem wichtig für Diagnose: Correlation(sales, transactions)

    Wenn:
		•	hohe Korrelation → gut modellierbar
		•	keine Korrelation → Zufallsrauschen

Holidays

	Fragen:
		•	Welche Feiertage haben echte Effekte?
		•	national vs regional vs local?



3. Wann ist eine Serie „forecastbar“?

Ich nutze in echten Projekten meist eine Scoring-Funktion:

Minimal-Score Beispiel

Für jede Store×Item-Serie:

Kriterium                          	 Score

	#Beobachtungen > 365                +2

	Sparsity > 0.6                      +2

	Varianz > Schwelle                  +1

	Keine Totphase > 60 Tage            +1

	Corr mit Transactions > 0.3         +2

	Item aktiv im letzten Monat         +2


Max: 10 Punkte
Cutoff:
	•	≥7 → forecastbar
	•	4–6 → risky
	•	<4 → nicht forecasten




In [ ]:
store_item_daily = load_table(PreDataset.STORE_ITEM_DAILY)

In [ ]:
store_item_daily.head()

In [ ]:
item_daily = load_table(PreDataset.ITEM_DAILY)

In [ ]:
item_daily.head()